# ⏳ TimeMeshin: Real-Time & Real-Data Ingestion Context Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Changmaulee/timemeshin/blob/main/examples/TimeMeshin_Colab_Quickstart.ipynb)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)
[![GitHub Stars](https://img.shields.io/github/stars/Changmaulee/timemeshin?style=social)](https://github.com/Changmaulee/timemeshin)

TimeMeshin is a **Spatio-Temporal ($S \times T$) Video-Scrubber Context Engine** for LLMs.
This notebook demonstrates **real-time ingestion of real-world multi-format data** (Live GitHub Commits, HTML, Markdown, Incident Logs, and User Uploads) with deterministic temporal playhead scrubbing.

In [ ]:
# Step 1: Install TimeMeshin and dependencies
!pip install git+https://github.com/Changmaulee/timemeshin.git
!pip install ipywidgets requests tabulate pypdf

## 🌐 Mode A: Ingest Live Real-World Data from the Web (GitHub API / Real Incidents)
Let's fetch live real-world commits and issue timelines from an active open-source repository (e.g., `fastapi` or `pytorch`) and stream them into TimeMeshin as temporal P-Frames.

In [ ]:
import requests
from datetime import datetime
from timemeshin import TimeMeshinClient
from tabulate import tabulate

# Initialize client
client = TimeMeshinClient(db_path='live_realdata.db')

# Fetch real public commits from GitHub API
repo = 'tiangolo/fastapi'
print(f'Fetching live real-world commit history from https://github.com/{repo} ...')
url = f'https://api.github.com/repos/{repo}/commits?per_page=20'
headers = {'Accept': 'application/vnd.github.v3+json'}
resp = requests.get(url, headers=headers)
commits = resp.json()

print(f'Ingesting {len(commits)} real-world git events into TimeMeshin...')
for c in reversed(commits):  # Ingest chronologically
    commit_data = c.get('commit', {})
    author = commit_data.get('author', {}).get('name', 'Unknown')
    date_str = commit_data.get('author', {}).get('date', '')[:19].replace('T', ' ')
    msg = commit_data.get('message', '').split('\n')[0][:80]
    sha = c.get('sha', '')[:7]
    
    try:
        ts = datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
    except Exception:
        ts = datetime.utcnow()
        
    # Ingest event delta into TimeMeshin
    client.ingest_event(
        timestamp=ts,
        rack='FastAPI_Repo',
        entity=f'Commit_{sha}',
        value=f'{author}: {msg}',
        reason=f'Git Push {sha}'
    )

print('✅ Real GitHub Live Stream successfully ingested into TimeMeshin timeline!')

## 📄 Mode B: Ingest Real Multi-Format Documents (.HTML, .MD, .PDF, .JSON)
Download and ingest real architectural documents and web pages using `DocumentLoader`.

In [ ]:
from timemeshin.ingestion.document_loader import DocumentLoader

# Download a real sample technical briefing HTML file
sample_url = 'https://raw.githubusercontent.com/Changmaulee/timemeshin/main/TIMEMESHIN_EXECUTIVE_SUMMARY.html'
r = requests.get(sample_url)
with open('real_doc.html', 'w', encoding='utf-8') as f:
    f.write(r.text)

# Universal Multi-Format Loader parses and cleans sections
doc_events = DocumentLoader.load_file('real_doc.html')
print(f'Extracted {len(doc_events)} clean structured events from HTML document!')

# Ingest into TimeMeshin timeline
for idx, ev in enumerate(doc_events[:15]):
    client.ingest_event(
        timestamp=ev['timestamp'],
        rack='ArchitectureSpec',
        entity=f'Section_{idx+1}',
        value=ev['text'][:70],
        reason='Document Section Ingestion'
    )

print('✅ Real Document Ingested & Spliced into Timeline!')

## 📤 Mode C: Upload Your Own Real File (Drag & Drop in Colab)
You can upload your own `.html`, `.md`, `.pdf`, `.json`, or `.txt` file directly into this Colab cell:

In [ ]:
try:
    from google.colab import files
    print('Upload your real file (.html, .md, .pdf, .json, .txt):')
    uploaded = files.upload()
    for fn in uploaded.keys():
        parsed_events = DocumentLoader.load_file(fn)
        print(f'Ingesting {len(parsed_events)} events from {fn} ...')
        for idx, ev in enumerate(parsed_events):
            client.ingest_event(
                timestamp=ev['timestamp'],
                rack='UserUpload',
                entity=f'Entry_{idx+1}',
                value=ev['text'][:80],
                reason=f'File Ingestion: {fn}'
            )
        print(f'✅ Successfully ingested {fn} into TimeMeshin!')
except Exception as e:
    print('Note: Drag-and-drop file upload is available when running directly in Google Colab (', e, ')')

## 🎬 Step 3: Interactive Video-Scrubber Playhead (Time Travel in Colab)
Drag the interactive slider below to travel backwards and forwards in time through your ingested real-world data.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Retrieve full chronological timeline
all_deltas = client.engine.delta_log
if not all_deltas:
    print('No events in timeline yet.')
else:
    max_idx = len(all_deltas) - 1
    
    def scrub_timeline(step_idx):
        target_event = all_deltas[step_idx]
        target_time = target_event.timestamp
        
        # Reconstruct ground truth at exactly this timestamp
        result = client.query_at(target_time, query='')
        
        print('=' * 75)
        print(f'⏱️ PLAYHEAD AT STEP {step_idx + 1}/{len(all_deltas)} | Timestamp: {target_time}')
        print('=' * 75)
        
        # Format State Matrix
        matrix_rows = []
        for k, v in result['state'].items():
            matrix_rows.append([k, v, 'COMMITTED'])
            
        if matrix_rows:
            print(tabulate(matrix_rows, headers=['Entity / Variable', 'Active Ground-Truth Value', 'Modality'], tablefmt='grid'))
        else:
            print('No active variables at this point.')
            
        print('\n🔗 Causal Domino Trail:')
        print(result['causal_summary'])
        
    slider = widgets.IntSlider(
        value=max_idx,
        min=0,
        max=max_idx,
        step=1,
        description='Playhead:',
        continuous_update=False,
        layout=widgets.Layout(width='650px')
    )
    widgets.interact(scrub_timeline, step_idx=slider)

## ⚡ Step 4: Natural Language Ground-Truth Query
Ask natural language questions at any specific historical timestamp.

In [ ]:
# Query state at the latest playhead
latest_time = all_deltas[-1].timestamp if all_deltas else datetime.utcnow()
res = client.query_at(latest_time, query='architecture commit status')

print(f'🔍 Query Result at {latest_time}:')
print(f'• Total Active Entities: {len(res["state"])}')
print(f'• Temporal Fence: Strictly 0% future data leakage')
print('\nActive State Sample:')
for k, v in list(res['state'].items())[:6]:
    print(f'  - {k} ➔ {v}')